# 🚀 Modern Formats Lab: PDBx, mmCIF & BinaryCIF
### Transitioning to the Future of Structural Biology Data

---

## 🎯 Overview

The legacy `.pdb` format is reaching its end-of-life. It cannot support structures with more than 99,999 atoms, more than 62 chains, or modern 12-character PDB IDs. To solve this, the community has moved to **mmCIF (PDBx)** and **BinaryCIF (BCIF)**.

**In this lab, you will explore:**
1. **Scalability**: Build a "Massive Molecule" that would crash a legacy PDB parser.
2. **Efficiency**: Use **BinaryCIF** for high-performance AI pipelines.
3. **Metadata**: Harvest rich database columns that PDB files simply cannot store.

> **Status Check**: As of 2027, the RCSB PDB has officially deprecated the legacy `.pdb` format for all new entries. Being "CIF-Native" is no longer optional for structural biologists.

In [ ]:
# [CONFIG] Setup & Installation
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("[INFO] Running in Google Colab - Installing dependencies...")
    %pip install -q synth-pdb biotite
else:
    print("[INFO] Running in local environment")
    sys.path.append(os.path.abspath("../../"))

from synth_pdb.generator import generate_pdb_content, PeptideResult

print("[OK] Modern Formats Lab Ready!")

## 🏗️ 1. Breaking the 99,999 Atom Wall

Legacy PDB files use a fixed-width format (column 7-11 for atom index). This means once you hit atom **100,000**, the column overflows and the file becomes unparseable. 

Let's generate a massive synthetic protein with ~150,000 atoms to see how mmCIF handles it effortlessly.

In [ ]:
# Generate a massive protein (approx 18,000 residues)
TEST_MODE = os.environ.get("SYNTH_PDB_TEST_MODE") == "1"
massive_len = 12500 if TEST_MODE else 18000
print(f"[BUILD] Building a massive {massive_len:,} residue peptide...")

# This returns a string in mmCIF format
cif_content = generate_pdb_content(length=massive_len, output_format="cif")

# Parse it back
result = PeptideResult(cif_content, format="cif")
n_atoms = result.structure.array_length()

print(f"[OK] Successfully generated {n_atoms:,} atoms!")
print(f"[DATA] Last atom index in CIF: {n_atoms}")

if n_atoms > 99999:
    print("\n🚨 This structure is PHYSICALLY IMPOSSIBLE to store in a legacy .pdb file.")

## ⚡ 2. BinaryCIF (BCIF): High-Performance AI

While mmCIF is great for humans, **BinaryCIF** is designed for machines. It uses binary compression (Delta-encoding, Run-length encoding) to reduce file size and dramatically speed up parsing.

This is the format used by the **Mol*** viewer and high-throughput AI pipelines.

In [ ]:
import time

# 1. Compare file sizes
test_len = 1000 if TEST_MODE else 5000
cif_text = generate_pdb_content(length=test_len, output_format="cif")
bcif_bin = generate_pdb_content(length=test_len, output_format="bcif")

print(f"[MEASURE] mmCIF Size: {len(cif_text) / 1024:.1f} KB")
print(f"[MEASURE] BinaryCIF Size: {len(bcif_bin) / 1024:.1f} KB")
print(f"[RESULT] Compression Ratio: {len(cif_text) / len(bcif_bin):.1f}x")

# 2. Compare Parsing Speed
start = time.time()
PeptideResult(cif_text, format="cif")
t_cif = time.time() - start

start = time.time()
PeptideResult(bcif_bin, format="bcif")
t_bcif = time.time() - start

print(f"\n[SPEED] mmCIF Parse: {t_cif:.4f}s")
print(f"[SPEED] BinaryCIF Parse: {t_bcif:.4f}s")
print(f"[RESULT] BinaryCIF is {t_cif / t_bcif:.1f}x faster to parse!")

## 📊 3. Accessing Rich Metadata

In mmCIF, everything is stored in categories (e.g., `_atom_site`, `_entity`, `_struct_conf`). This allows us to store complex data like **pLDDT confidence scores** or **Experimental Errors** directly in the file without hacking the 'B-factor' column.

In [ ]:
# Look at the raw mmCIF header
print("[DATA] Sample of mmCIF Category Headers:")
header_lines = [l for l in cif_content.splitlines() if l.startswith("_")][:15]
for l in header_lines:
    print(f"  {l}")

print("\n[INFO] Note how every column is explicitly named.")
print("No more counting characters to find the X-coordinate!")

---

## 🎓 Summary

1. **Scalability**: mmCIF supports molecules of arbitrary size.
2. **Speed**: BinaryCIF is the standard for high-performance AI and web visualization.
3. **Predictability**: Dictionary-based parsing is much more robust than fixed-width text parsing.

**The structural biology world has moved on from the 1970s PDB format. Now, your code has too.**